# Baseline Evaluation — Perplexity Before Fine-Tuning

Runs the full test split (1,502 examples) through `run_benchmark()` to produce `results/baseline_results.json` — the "before" side of the before/after comparison required for Week 1.

**Before running:** on Kaggle, enable **Settings → Accelerator → GPU T4 x2**. Do not pick P100 — modern PyTorch wheels have dropped kernel support for the P100's older `sm_60` architecture.

## 1. Pull the repo and install dependencies

This repo is private. Before running:

1. Create a GitHub Personal Access Token (fine-grained, scoped to just this repo, **Contents: Read-only**).
2. In this Kaggle notebook: **Add-ons → Secrets → Add a new secret**, name it `GITHUB_TOKEN`.
3. Run the cell below — it reads the secret and never prints it.

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
repo_url = f"https://{github_token}@github.com/zoom-BT/llm-alignment-internship.git"

!git clone {repo_url}
%cd llm-alignment-internship
!pip install -q -r requirements.txt

## 2. Confirm the GPU is visible

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

## 3. Run the baseline benchmark

`run_benchmark` loads `config['model']['base_model_name']`, moves it to whatever device `get_device()` finds (the T4 here, since we're not on the CPU-only local machine anymore), computes perplexity on the full test split, and writes `results/baseline_results.json` itself.

This was measured locally at ~35s for 20 examples on CPU (~45 min projected for all 1,502) — on a T4 this should take a small fraction of that.

In [ ]:
import yaml
from src.evaluate import run_benchmark

config = yaml.safe_load(open("config.yaml"))
results = run_benchmark(config)
results

## 4. Bring the results back into the local repo

Kaggle notebooks don't push to GitHub. Download `results/baseline_results.json` from the Kaggle output panel (after a **Save Version** run — the live-session file browser won't show it otherwise), drop it into the local `results/` folder, and commit it there.